In [68]:
import warnings
from typing import Literal

from typing import Sequence
import folium
from folium.plugins import HeatMap
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [56]:
LOCATION_CENTRO_BRASIL = [-15, -55]
LOCATION_CENTRO_CIDADE_SP = [-23.6307, -46.6334]

In [98]:
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)

In [99]:
def gerar_mapa_de_calor(
    data: pd.DataFrame,
    lat_label: str,
    long_label: str,
    location: Sequence[float],
    zoom_start: int = 10,
    tiles: str = "Esri.WorldTopoMap",
    heat_radius: int = 25,
    heat_blur: int = 15,
) -> folium.Map:
    mapa = folium.Map(location, zoom_start=zoom_start, tiles=tiles)

    coordenadas = data[[lat_label, long_label]].dropna().to_dict()
    latitudes = coordenadas[lat_label]
    longitudes = coordenadas[long_label]

    heat_data = list(zip(latitudes.values(), longitudes.values()))

    mapa_de_calor = HeatMap(heat_data, radius=heat_radius, blur=heat_blur)

    mapa.add_child(mapa_de_calor)

    return mapa

In [130]:
def gerar_grafico_top(
    data: pd.DataFrame,
    tipo: Literal["barras", "colunas"],
    *,
    category_label: str,
    title: str,
    n_top: int | None = None,
    count_label: str = "Quantidade",
    alt_category_label: str | None = None,
) -> None:
    if n_top is None:
        n_top = len(data[category_label].unique())

    if alt_category_label is None:
        alt_category_label = category_label

    data_top = (
        data[category_label]
        .dropna()
        .value_counts()
        .head(n_top)
        .reset_index()
        .rename(columns={"count": count_label})
    )

    data_top[category_label] = pd.Categorical(
        data_top[category_label],
        categories=data_top[category_label].tolist(),
        ordered=True,
    )

    plt.figure(figsize=(12, 6))

    match tipo:
        case "barras":
            x = count_label
            y = category_label
            xlabel = count_label
            ylabel = alt_category_label

            max_value = data_top[count_label].max()
            plt.xlim(0, max_value * 1.12)

        case "colunas":
            x = category_label
            y = count_label
            xlabel = alt_category_label
            ylabel = count_label

            max_value = data_top[count_label].max()
            plt.ylim(0, max_value * 1.12)

    ax = sns.barplot(
        data_top,
        x=x,
        y=y,
        palette="viridis",
        legend=False,
    )

    for bar in ax.containers:
        ax.bar_label(bar, padding=3)

    plt.ylabel(xlabel)
    plt.xlabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Análise População em Situação de Rua (São Paulo, SP)

In [134]:
df_populacao_rua = pd.read_excel("data/a_censo-base-de-dados-entrega_ods.ods", engine="odf")

In [ ]:
df_populacao_rua.head()

In [ ]:
mapa_sinistros_de_transito = gerar_mapa_de_calor(
    df_populacao_rua,
    lat_label="Latitude",
    long_label="Longitude",
    location=LOCATION_CENTRO_CIDADE_SP,
    zoom_start=11,
    heat_radius=8,
    heat_blur=10,
)

mapa_sinistros_de_transito

In [ ]:
gerar_grafico_top(
    df_populacao_rua,
    n_top=10,
    category_label="Distrito",
    title="Top 10 Distritos com Mais Moradores de Rua",
    count_label="Número de Moradores de Rua",
    alt_category_label="Distrito",
)

In [ ]:
gerar_grafico_top(
    df_populacao_rua,
    tipo="barras",
    n_top=10,
    category_label="Faixa de idade",
    title="Top 10 Faixas de Idade de Moradores de Rua",
    count_label="Número de Moradores de Rua",
    alt_category_label="Faixa de Idade",
)

In [ ]:
gerar_grafico_top(
    df_populacao_rua,
    tipo="colunas",
    category_label="Sexo",
    title="Top Faixas de Idade de Moradores de Rua",
    count_label="Número de Moradores de Rua",
    alt_category_label="Faixa de Idade",
)

# Análise Criminalidade (Estado SP)

In [ ]:
df_criminalidade = pd.concat(
    [
        pd.read_excel(
            "data/SPDadosCriminais_2026.xlsx", sheet_name="JAN-MAR_2026"
        ),
        pd.read_excel(
            "data/SPDadosCriminais_2025.xlsx", sheet_name="JAN-JUN_2025"
        ),
        pd.read_excel(
            "data/SPDadosCriminais_2025.xlsx", sheet_name="JUL-DEZ_2025"
        ),
    ]
)

In [ ]:
df_criminalidade.head()

In [ ]:
mapa_sinistros_de_transito = gerar_mapa_de_calor(
    df_populacao_rua,
    lat_label="Latitude",
    long_label="Longitude",
    location=LOCATION_CENTRO_CIDADE_SP,
    zoom_start=11,
    heat_radius=8,
    heat_blur=10,
)

mapa_sinistros_de_transito